In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
df= spark.read.table("samples.bakehouse.sales_transactions")
df.display()

In [0]:
# to check the schema of df
df.printSchema()

In [0]:
#df2 =df.where("paymentMethod =='mastercard' ")
df2 = df.filter(col("paymentMethod")== 'mastercard')
#df2 = df.select(expr("paymentMethod == 'mastercard'").alias("isMasterCard")) this is wrong
display(df2)

In [0]:
#check the df2 schema after using filter 
df2.printSchema()

In [0]:

df2 = df.filter(col("paymentMethod") != 'mastercard')
display(df2)

In [0]:
%sql
--select * from  samples.bakehouse.sales_transactions where paymentMethod = 'mastercard';
select * from  samples.bakehouse.sales_transactions where paymentMethod <> 'mastercard';

In [0]:
#df2=df.withColumn("power",pow("quantity",2)/6).display()
df2 = df.withColumn("power",round(pow("quantity",2)/6,2))
#display(df2)
df2.printSchema()

In [0]:
display(df2)
#df2.printSchema()

In [0]:
%sql
select * ,round(pow(quantity,2/6),2) as power from samples.bakehouse.sales_transactions where paymentMethod = 'mastercard'  limit 10;

In [0]:
df3=df.select("product")
df3.groupBy("product").count().filter(col("product") =="Golden Gate Ginger"). display()

In [0]:
df3= df.select("product").distinct()
df3.display()

In [0]:
df4=df3.withColumn("lower_product",expr("lower(product)"))\
    .withColumn("upper_product",expr("upper(product)"))\
    .withColumn("trim_product",expr("trim(product)"))\
    .withColumn("substring_product",expr("substr(product,0,4)"))\
    .withColumn("split_product",expr("split(product,' ')"))
display(df4)    
                        


In [0]:
df5=df3.withColumn("lower_product",lower(col("product")))\
    .withColumn("upper_product",upper(col("product")))\
    .withColumn("trim_product",trim(col("product")))\
    .withColumn("substring_product",substring(col("product"),0,4))\
    .withColumn("split_product",split(col("product")," "))
    
display(df5) 

In [0]:
%sql
select product, lower(product) as lower_product,
upper(product) as upper_product,
trim(product) as trim_product,
substr(product,1,4) as substring_product,
split(product,' ') as split_product
from samples.bakehouse.sales_transactions;


In [0]:
df6 =df.select("dateTime").withColumn("date", current_date())\
    .withColumn("time",current_time())\
    .withColumn("time_stamp",current_timestamp())\
    .withColumn("datetime_date",to_date("datetime"))\
    .withColumn("date_add",date_add(to_date("datetime"),5))\
    .withColumn("date_sub",date_sub(to_date("datetime"),5))\
    .withColumn("constant_datetime",lit(current_date()))
display(df6)

In [0]:
df7 =df.select("dateTime").withColumn("date", current_date())\
    .withColumn("time",current_time())\
    .withColumn("time_stamp",current_timestamp())\
    .withColumn("datetime_date",current_date())\
    .withColumn("date_add",date_add(col("datetime"),5))\
    .withColumn("date_sub",date_sub(col("datetime"),5))\
    .withColumn("constant_datetime",lit(current_date()))\
    .withColumn("custom_date",to_date(lit('20260910' ),'yyyyMMdd'))
display(df7)

In [0]:
%sql
select datetime,date_add(datetime,5) as date_add,date_sub(datetime,5) as date_sub,
date_diff(datetime,current_date()) as date_difference,month(datetime) as monthname
from samples.bakehouse.sales_transactions limit 10;

In [0]:
df2= df.select("transactionID","customerID") \
        .withColumn("coalesce_id",coalesce("transactionId","customerId")) \
        .withColumn("if_else",expr("case when transactionId is NULL then customerId else transactionId end")) \
        .withColumn("when_condition",when(col("transactionID")== 0 ,col("customerId")).otherwise(col("transactionID")))\
.na.drop(subset=["transactionID"])
# .na.drop('all')
# .na.fill('myvalue')
display(df2)  
df2.limit(10).display()
##.filter((col("customerId") ==  0)  | (col("transactionID") == 0))

In [0]:
%sql
select transactionId,customerId,
coalesce(transactionId,customerId) as coalesce_id,case when transactionId is null then customerId else transactionId end as case_statement
from samples.bakehouse.sales_transactions
limit 10

In [0]:
%sql
select customerId ,count(customerId)from samples.bakehouse.sales_transactions
group By customerId
having count(customerId) >4 
order by count(customerId) desc

In [0]:
%sql
--update samples.bakehouse.sales_transactions set customerId = 0
--where customerId = 2000029

In [0]:
import time
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

# 1. Define your transformation logic (Lazy)
df_clean = df.filter(df["quantity"] > 0)

# 2. Capture metrics using lightweight eager actions (No data rows are returned)
start_time = time.time()
inserted_rows = df_clean.count()  # Returns a simple integer, NOT a full dataframe
end_time = time.time()

# 3. Create a single-row metadata DataFrame
audit_data = [(
    "sales_pipeline", 
    "SUCCESS", 
    inserted_rows, 
    round(end_time - start_time, 2)
)]

audit_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("duration_seconds", StringType(), True)
])

df_audit = spark.createDataFrame(audit_data, audit_schema)

# 4. Save metrics securely to your infrastructure logs
df_audit.write.format("delta").mode("append").saveAsTable("samples.bakehouse.pipeline_logs")
